# Example 1.2.3 / 3: Descriptive Names

Companion notebook for **1.2.3 Symbolic and Logical Parameters** in the AMPL lab Documentation, section "Descriptive Names". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Descriptive Names

Symbolic parameters can attach a longer description to each member of a set. The short code
stays the set member; the description is used only in reports. From Python, the dictionary's
values are the strings. `to_dict()` returns them keyed by member, and the set's `members()`
lists the members in their order, for a report:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;
param orig_name {ORIG} symbolic;
param supply {ORIG} >= 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIG"] = ["GARY", "CLEV", "PITT"]
ampl.param["orig_name"] = {"GARY": "Gary, Indiana", "CLEV": "Cleveland, Ohio",
                           "PITT": "Pittsburgh, Pennsylvania"}
ampl.param["supply"] = {"GARY": 1400, "CLEV": 2600, "PITT": 2900}

In [5]:
# run.py
name = ampl.param["orig_name"].to_dict()
supply = ampl.param["supply"].to_dict()
for i in ampl.set["ORIG"].members():
    print(f"{name[i]:<26} {supply[i]:6d}")

Gary, Indiana                1400
Cleveland, Ohio              2600
Pittsburgh, Pennsylvania     2900


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. `printf` writes the report:

In [6]:
%%writefile data.dat
set ORIG := GARY CLEV PITT;
param:  orig_name                    supply :=
  GARY  "Gary, Indiana"               1400
  CLEV  "Cleveland, Ohio"             2600
  PITT  "Pittsburgh, Pennsylvania"    2900 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
printf {i in ORIG} "%-26s %6d\n", orig_name[i], supply[i];

Gary, Indiana                1400
Cleveland, Ohio              2600
Pittsburgh, Pennsylvania     2900
